In [3]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from utils.knowledge_check import knowledge_check_truthfulqa, knowledge_check_mmlu
from utils.generation import generate_response, NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO
# from utils.activation import extract_activations
from utils.probe import probe_all_layers, train_linear_probe
import warnings
warnings.filterwarnings("ignore")   # suppress the greedy-decoding flag warnings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DATA_DIR = Path("data/dataset")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

Device: cuda
GPU: NVIDIA GeForce RTX 4090
VRAM: 25.4 GB


In [4]:
deception_df = pd.read_csv(DATA_DIR / "deception_dataset.csv")
print(f"deception_dataset: {deception_df.shape}")
print(deception_df["label"].value_counts())

deception_dataset: (400, 6)
label
honest       200
deceptive    200
Name: count, dtype: int64


In [5]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

N_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded: {N_LAYERS} layers, hidden_dim={HIDDEN_DIM}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded: 28 layers, hidden_dim=3584


In [6]:
tqa_mc = load_dataset("truthful_qa", "multiple_choice", split="validation")
TRUTHFULQA_PATH = DATA_DIR / "truthfulQA_test_results.csv"
CHECKPOINT_EVERY = 50

if TRUTHFULQA_PATH.exists():
    kc_df = pd.read_csv(TRUTHFULQA_PATH)
    if len(kc_df) == len(tqa_mc):
        print(f"Already complete ({len(kc_df)} rows). Loading from {TRUTHFULQA_PATH}")
    else:
        done_questions = set(kc_df["question"].tolist())
        remaining = [item for item in tqa_mc if item["question"] not in done_questions]
        print(f"Resuming from checkpoint: {len(kc_df)} done, {len(remaining)} remaining")
        records = []
        for i, item in enumerate(tqdm(remaining, desc="TruthfulQA knowledge check")):
            records.append(knowledge_check_truthfulqa(item, model, tokenizer, DEVICE))
            if (i + 1) % CHECKPOINT_EVERY == 0:
                pd.DataFrame(records).to_csv(TRUTHFULQA_PATH, mode="a", header=False, index=False)
                records = []
        if records:
            pd.DataFrame(records).to_csv(TRUTHFULQA_PATH, mode="a", header=False, index=False)
        kc_df = pd.read_csv(TRUTHFULQA_PATH)
        print(f"Done. Total: {len(kc_df)}")
else:
    records = []
    for i, item in enumerate(tqdm(tqa_mc, desc="TruthfulQA knowledge check")):
        records.append(knowledge_check_truthfulqa(item, model, tokenizer, DEVICE))
        if (i + 1) % CHECKPOINT_EVERY == 0:
            df_chunk = pd.DataFrame(records)
            df_chunk.to_csv(TRUTHFULQA_PATH, mode="a", header=not TRUTHFULQA_PATH.exists(), index=False)
            records = []
    if records:
        pd.DataFrame(records).to_csv(TRUTHFULQA_PATH, mode="a", header=not TRUTHFULQA_PATH.exists(), index=False)
    kc_df = pd.read_csv(TRUTHFULQA_PATH)
    print(f"Done. Total: {len(kc_df)}  |  Passed: {kc_df['passed'].sum()}  |  Failed: {(~kc_df['passed']).sum()}")

passed_df = kc_df[kc_df["passed"] == True].reset_index(drop=True)
failed_df = kc_df[kc_df["passed"] == False].reset_index(drop=True)
print(f"Passed: {len(passed_df)}  |  Failed: {len(failed_df)}")

Already complete (817 rows). Loading from data/dataset/truthfulQA_test_results.csv
Passed: 398  |  Failed: 419


In [7]:
mmlu = load_dataset("cais/mmlu", "all", split="test")
MMLU_PATH = DATA_DIR / "mmlu_test_results.csv"
CHECKPOINT_EVERY = 50

if MMLU_PATH.exists():
    mmlu_kc_df = pd.read_csv(MMLU_PATH)
    if len(mmlu_kc_df) == len(mmlu):
        print(f"Already complete ({len(mmlu_kc_df)} rows). Loading from {MMLU_PATH}")
    else:
        done_questions = set(mmlu_kc_df["question"].tolist())
        remaining = [item for item in mmlu if item["question"] not in done_questions]
        print(f"Resuming from checkpoint: {len(mmlu_kc_df)} done, {len(remaining)} remaining")
        records = []
        for i, item in enumerate(tqdm(remaining, desc="MMLU knowledge check")):
            records.append(knowledge_check_mmlu(item, model, tokenizer, DEVICE))
            if (i + 1) % CHECKPOINT_EVERY == 0:
                pd.DataFrame(records).to_csv(MMLU_PATH, mode="a", header=False, index=False)
                records = []
        if records:
            pd.DataFrame(records).to_csv(MMLU_PATH, mode="a", header=False, index=False)
        mmlu_kc_df = pd.read_csv(MMLU_PATH)
        print(f"Done. Total: {len(mmlu_kc_df)}")
else:
    records = []
    for i, item in enumerate(tqdm(mmlu, desc="MMLU knowledge check")):
        records.append(knowledge_check_mmlu(item, model, tokenizer, DEVICE))
        if (i + 1) % CHECKPOINT_EVERY == 0:
            df_chunk = pd.DataFrame(records)
            df_chunk.to_csv(MMLU_PATH, mode="a", header=not MMLU_PATH.exists(), index=False)
            records = []
    if records:
        pd.DataFrame(records).to_csv(MMLU_PATH, mode="a", header=not MMLU_PATH.exists(), index=False)
    mmlu_kc_df = pd.read_csv(MMLU_PATH)
    print(f"Done. Total: {len(mmlu_kc_df)}  |  Passed: {mmlu_kc_df['passed'].sum()}  |  Failed: {(~mmlu_kc_df['passed']).sum()}")

mmlu_passed_df = mmlu_kc_df[mmlu_kc_df["passed"] == True].reset_index(drop=True)
mmlu_failed_df = mmlu_kc_df[mmlu_kc_df["passed"] == False].reset_index(drop=True)
print(f"Passed: {len(mmlu_passed_df)}  |  Failed: {len(mmlu_failed_df)}")

Resuming from checkpoint: 14038 done, 0 remaining


MMLU knowledge check: 0it [00:00, ?it/s]

Done. Total: 14038
Passed: 6596  |  Failed: 7442


In [8]:
TRUTHFULQA_RESPONSES_PATH = DATA_DIR / "truthfulQA_responses.csv"
CHECKPOINT_EVERY = 50

configs = [
    ("A", passed_df, NEUTRAL_SYSTEM),
    ("B", failed_df, NEUTRAL_SYSTEM),
    ("C", passed_df, FACTUAL_DECEPTION_SCENARIO),
]

if TRUTHFULQA_RESPONSES_PATH.exists():
    resp_df = pd.read_csv(TRUTHFULQA_RESPONSES_PATH)
    done_keys = set(zip(resp_df["question"], resp_df["config"]))
    total_expected = len(passed_df) * 2 + len(failed_df)
    if len(resp_df) == total_expected:
        print(f"Already complete ({len(resp_df)} rows). Loading from {TRUTHFULQA_RESPONSES_PATH}")
    else:
        print(f"Resuming from checkpoint: {len(resp_df)} done, {total_expected - len(resp_df)} remaining")
else:
    done_keys = set()
    resp_df = None
    print("Starting fresh.")

for config_name, source_df, system_prompt in configs:
    remaining = source_df[~source_df["question"].isin(
        {q for q, c in done_keys if c == config_name}
    )].reset_index(drop=True)

    if len(remaining) == 0:
        print(f"Config {config_name}: already complete, skipping.")
        continue

    print(f"Config {config_name}: {len(remaining)} rows to generate.")
    records = []
    for i, row in enumerate(tqdm(remaining.itertuples(), total=len(remaining), desc=f"Config {config_name}")):
        records.append({
            "question": row.question,
            "correct_answer": row.correct_answer,
            "config": config_name,
            "system_prompt": system_prompt,
            "response": generate_response(row.question, model, tokenizer, DEVICE, system_prompt=system_prompt),
        })
        if (i + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(records).to_csv(
                TRUTHFULQA_RESPONSES_PATH, mode="a",
                header=not TRUTHFULQA_RESPONSES_PATH.exists(), index=False
            )
            done_keys.update((r["question"], r["config"]) for r in records)
            records = []
    if records:
        pd.DataFrame(records).to_csv(
            TRUTHFULQA_RESPONSES_PATH, mode="a",
            header=not TRUTHFULQA_RESPONSES_PATH.exists(), index=False
        )

resp_df = pd.read_csv(TRUTHFULQA_RESPONSES_PATH)
print(f"\nDone. Total rows: {len(resp_df)}")
print(resp_df["config"].value_counts())

Starting fresh.
Config A: 398 rows to generate.


Config A:   0%|          | 0/398 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Config B: 419 rows to generate.


Config B:   0%|          | 0/419 [00:00<?, ?it/s]

Config C: 398 rows to generate.


Config C:   0%|          | 0/398 [00:00<?, ?it/s]


Done. Total rows: 1215
config
B    419
A    398
C    398
Name: count, dtype: int64


In [9]:
MMLU_RESPONSES_PATH = DATA_DIR / "mmlu_responses.csv"
CHECKPOINT_EVERY = 50

configs = [
    ("A", mmlu_passed_df, NEUTRAL_SYSTEM),
    ("B", mmlu_failed_df, NEUTRAL_SYSTEM),
    ("C", mmlu_passed_df, FACTUAL_DECEPTION_SCENARIO),
]

if MMLU_RESPONSES_PATH.exists():
    mmlu_resp_df = pd.read_csv(MMLU_RESPONSES_PATH)
    done_keys = set(zip(mmlu_resp_df["question"], mmlu_resp_df["config"]))
    total_expected = len(mmlu_passed_df) * 2 + len(mmlu_failed_df)
    if len(mmlu_resp_df) == total_expected:
        print(f"Already complete ({len(mmlu_resp_df)} rows). Loading from {MMLU_RESPONSES_PATH}")
    else:
        print(f"Resuming from checkpoint: {len(mmlu_resp_df)} done, {total_expected - len(mmlu_resp_df)} remaining")
else:
    done_keys = set()
    mmlu_resp_df = None
    print("Starting fresh.")

for config_name, source_df, system_prompt in configs:
    remaining = source_df[~source_df["question"].isin(
        {q for q, c in done_keys if c == config_name}
    )].reset_index(drop=True)

    if len(remaining) == 0:
        print(f"Config {config_name}: already complete, skipping.")
        continue

    print(f"Config {config_name}: {len(remaining)} rows to generate.")
    records = []
    for i, row in enumerate(tqdm(remaining.itertuples(), total=len(remaining), desc=f"Config {config_name}")):
        records.append({
            "question": row.question,
            "correct_answer": row.correct_answer,
            "config": config_name,
            "system_prompt": system_prompt,
            "response": generate_response(row.question, model, tokenizer, DEVICE, system_prompt=system_prompt),
        })
        if (i + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(records).to_csv(
                MMLU_RESPONSES_PATH, mode="a",
                header=not MMLU_RESPONSES_PATH.exists(), index=False
            )
            done_keys.update((r["question"], r["config"]) for r in records)
            records = []
    if records:
        pd.DataFrame(records).to_csv(
            MMLU_RESPONSES_PATH, mode="a",
            header=not MMLU_RESPONSES_PATH.exists(), index=False
        )

mmlu_resp_df = pd.read_csv(MMLU_RESPONSES_PATH)
print(f"\nDone. Total rows: {len(mmlu_resp_df)}")
print(mmlu_resp_df["config"].value_counts())

Starting fresh.
Config A: 6596 rows to generate.


Config A:   0%|          | 0/6596 [00:00<?, ?it/s]

Config B: 7442 rows to generate.


Config B:   0%|          | 0/7442 [00:00<?, ?it/s]

Config C: 6596 rows to generate.


Config C:   0%|          | 0/6596 [00:00<?, ?it/s]


Done. Total rows: 20634
config
B    7442
A    6596
C    6596
Name: count, dtype: int64
